# Run all batches

Delivers the 4 batches **one at a time**, the way a sender would: write the `_SUCCESS` marker last.
EventBridge starts the workflow by itself. This notebook only writes the marker and waits for the result.
Then it checks the gold tables and the KPI views.

One at a time, because every run merges into the same Iceberg tables (see `docs/phase_07_step_functions`).

## 1. Setup

In [1]:
import sys
from pathlib import Path

# Works whether Jupyter starts in the project root or in notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "infra"))

import sql_files
from aws_helpers import REGION, SETTINGS, account_id, aws_json, files_in_batch, release_batch, run_athena, whoami

BATCHES = range(1, SETTINGS["number_of_batches"] + 1)
MACHINE_ARN = f"arn:aws:states:{REGION}:{account_id()}:stateMachine:{SETTINGS['state_machine_name']}"
GOLD = SETTINGS["glue_databases"]["gold"]

print("identity:", whoami())
for batch in BATCHES:
    print(f"batch {batch}: {len(files_in_batch(batch))} data files in landing/batch_{batch:02d}/")

identity: user/Ahmed


batch 1: 37 data files in landing/batch_01/


batch 2: 34 data files in landing/batch_02/


batch 3: 34 data files in landing/batch_03/


batch 4: 34 data files in landing/batch_04/


## 2. Release the batches
For each batch: write the marker, wait for the run, print how it ended. Each run takes about 9 minutes (most of it Athena
checking once a minute whether a merge has finished). A run that does not end `SUCCEEDED` stops the notebook.
Releasing a batch that was loaded before is fine: every step overwrites its own result.

In [2]:
import time

runs = {}
for batch in BATCHES:
    started = time.time()
    run = release_batch(batch, MACHINE_ARN)
    runs[batch] = run
    print(f"batch {batch}: {run['status']:<10} {(time.time() - started) / 60:4.1f} min   output {run.get('output')}")
    assert run["status"] == "SUCCEEDED", (run.get("error"), run.get("cause"))
print()
print("OK  all", len(runs), "batches ran through the event trigger")

batch 1: SUCCEEDED   8.5 min   output {"file_count":37,"batch":1,"ingest_date":"2026-11-01"}


batch 2: SUCCEEDED   8.3 min   output {"file_count":34,"batch":2,"ingest_date":"2026-11-02"}


batch 3: SUCCEEDED   8.3 min   output {"file_count":34,"batch":3,"ingest_date":"2026-11-03"}


batch 4: SUCCEEDED   7.5 min   output {"file_count":34,"batch":4,"ingest_date":"2026-11-04"}

OK  all 4 batches ran through the event trigger


## 3. The gold tables

In [3]:
def count(table):
    return int(run_athena(f"SELECT COUNT(*) FROM {GOLD}.{table}")["rows"][0][0])


GOLD_COUNTS = {table: count(table) for table in ("dim_patient", "dim_facility", "dim_practitioner", "dim_date", "fact_encounter")}
for table, rows in GOLD_COUNTS.items():
    print(f"{table:<18}{rows:>8}")

classes = run_athena(f"SELECT encounter_class, COUNT(*) FROM {GOLD}.fact_encounter GROUP BY 1 ORDER BY 2 DESC")["rows"]
print()
print("encounters by class:", {name: int(rows) for name, rows in classes})

# Numbers established independently from the source files in Phase 6
assert GOLD_COUNTS["dim_patient"] == 137 and GOLD_COUNTS["dim_facility"] == 521 and GOLD_COUNTS["dim_practitioner"] == 521
assert GOLD_COUNTS["fact_encounter"] == 4586
assert {name: int(rows) for name, rows in classes} == {"AMB": 4313, "EMER": 127, "IMP": 114, "VR": 18, "HH": 14}
print("OK  the counts equal the source data: 137 patients, 521 facilities, 521 doctors, 4586 encounters")

dim_patient            137
dim_facility           521
dim_practitioner       521
dim_date             47847
fact_encounter        4586



encounters by class: {'AMB': 4313, 'EMER': 127, 'IMP': 114, 'VR': 18, 'HH': 14}
OK  the counts equal the source data: 137 patients, 521 facilities, 521 doctors, 4586 encounters


## 4. The data-quality checks, for every batch

In [4]:
CHECKS = sql_files.read_checks(ROOT / "sql" / "dq")
DETAILED = sql_files.detailed_checks_query(list(CHECKS.values()))
for batch in BATCHES:
    ingest_date = f"2026-11-{batch:02d}"
    result = run_athena(DETAILED, parameters=[sql_files.quote_date(ingest_date)], timeout=300)
    failing = {name: int(rows) for name, rows in result["rows"] if int(rows)}
    print(f"batch {batch} ({ingest_date}): {len(result['rows'])} checks, failing: {failing or 'none'}")
    assert not failing
print("OK  every check passes for every batch")

batch 1 (2026-11-01): 14 checks, failing: none


batch 2 (2026-11-02): 14 checks, failing: none


batch 3 (2026-11-03): 14 checks, failing: none


batch 4 (2026-11-04): 14 checks, failing: none
OK  every check passes for every batch


## 5. The five KPI views

In [5]:
VIEWS = {
    "vw_daily_admissions_discharges": "SUM(admissions), SUM(discharges)",
    "vw_avg_los_by_facility_month": "SUM(inpatient_stays), ROUND(SUM(avg_los_days * inpatient_stays) / SUM(inpatient_stays), 2)",
    "vw_readmission_30d": "SUM(inpatient_admissions), SUM(readmissions_30d)",
    "vw_ed_volume_by_hour_dow": "SUM(emergency_visits), COUNT(*)",
    "vw_top_reasons": "COUNT(*), MAX(encounters)",
}
VALUES = {}
for view, expression in VIEWS.items():
    VALUES[view] = run_athena(f"SELECT {expression} FROM {GOLD}.{view}")["rows"][0]
    print(f"{view:<34}{VALUES[view]}")

assert [int(v) for v in VALUES["vw_readmission_30d"]] == [114, 17]
assert int(VALUES["vw_ed_volume_by_hour_dow"][0]) == 127
print("OK  114 inpatient stays, 17 of them readmissions within 30 days; 127 emergency visits")

vw_daily_admissions_discharges    ['114', '114']


vw_avg_los_by_facility_month      ['114', '6.22']


vw_readmission_30d                ['114', '17']


vw_ed_volume_by_hour_dow          ['127', '78']


vw_top_reasons                    ['109', '498']
OK  114 inpatient stays, 17 of them readmissions within 30 days; 127 emergency visits


## 6. The runs, as Step Functions shows them

In [6]:
for e in aws_json("stepfunctions", "list-executions", "--state-machine-arn", MACHINE_ARN, "--max-results", "6")["executions"]:
    print(f"{e['startDate'][:19]}  {e['status']:<10}  {e['name'][:40]}")
print()
print(f"Step Functions console > {SETTINGS['state_machine_name']} > Executions. Charts of the KPIs: notebooks/kpi_analysis.ipynb")

2026-10-27T01:16:31  SUCCEEDED   334e3c36-b8fd-a826-255e-c1ef2efee5cd_ae1
2026-10-27T01:08:14  SUCCEEDED   3028e410-f88f-3da7-2b5e-ac3cc4817bfe_13b
2026-10-27T00:59:58  SUCCEEDED   2cf2ed42-2935-b226-08c7-2f7d121a2035_533
2026-10-27T00:51:30  SUCCEEDED   e7055472-0d9c-06dc-e1cd-bd69ee812989_fb5

Step Functions console > pf-batch-pipeline > Executions. Charts of the KPIs: notebooks/kpi_analysis.ipynb


## What this is for
| Item | Where |
|---|---|
| Deliver a batch | write `landing/batch_NN/_SUCCESS` (`release_batch` in `infra/aws_helpers.py` does it and waits) |
| Run one batch again | write its marker again; the result is the same |
| Look at the results | `kpi_analysis.ipynb` (charts), Athena console (`pf_gold` views) |
| Next | Phase 11 moves the AWS resources into CloudFormation; the same notebook then re-runs the batches on the new resources |